In [1]:
import tiktoken
import numpy as np
# from senten

In [2]:
def softmax(x, axis= 0):
    # subtract max for numerical stability

    # across rows
    x_shifted = x - np.max(x, axis = axis, keepdims=True)
    
    exp_x = np.exp(x_shifted)
    
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [3]:
print(softmax([[1,2,3000]], axis= 0)) # column
print(softmax([[1,2,3000]], axis= 1)) # row

[[1. 1. 1.]]
[[0. 0. 1.]]


In [4]:
def self_attention(query , key, value):
    # q, k, v are np arrays and are of same order
    # writing codea asuming they are horizontal matrices

    a = query @ key.T # dot product but for a number of words will be orchestrsted by cross product

    a = softmax(a , axis = 1)
    
    b = a @ value
    
    return b
    

In [5]:

q = np.array([[1,2,3],
              [2,1,3],
              [2,1,2], # here n = 4
              [4,3,2]]) # n x 3
k = np.array([[3,2,3],
              [2,3,4], 
              [2,1,3],
              [4,3,1]]) # n x 3

# n x n

v = np.array([[2,2,1],
              [3,2,1],
              [3,2,3],
              [4,3,1]]) # n x 3

# n x 3

In [6]:
test_embedding_of_words  = np.array([[1,2,3,4,5],
                                [2,1,3,5,2],
                                [4,3,3,2,2]]) # n x 5
# produce query, k ,v  vectors from embedding vector
test_wq = np.random.randn(5,5)  # 5 x 5
test_wk = np.random.randn(5,5)  # 5 x 5
test_wv = np.random.randn(5,5)  # 5 x 5

test_k = test_embedding_of_words @ test_wk   # n x 5
test_v = test_embedding_of_words @ test_wv   # n x 5
test_q = test_embedding_of_words @ test_wq   # n x 5


In [7]:
def generate_q_k_v(embedding_of_words, embedding_dimensions = 512, order_of_weigths = 64):
    wq = np.random.randn(embedding_dimensions,order_of_weigths)  # 5 x 5
    wk = np.random.randn(embedding_dimensions,order_of_weigths)  # 5 x 5
    wv = np.random.randn(embedding_dimensions,order_of_weigths)  # 5 x 5

    k = embedding_of_words @ wk   # n x 5
    v = embedding_of_words @ wv   # n x 5
    q = embedding_of_words @ wq   # n x 5
    return q, k, v

In [8]:
def multi_head_attnetion(embedding_of_words):
    heads = []
    for i in range(8):
        q,k,v = generate_q_k_v(embedding_of_words)
        head = self_attention(q,k,v)
        heads.append(head)
        # print(np.shape(head))

    context_aware =  np.concatenate(heads, axis = 1)
    # print(context_aware.shape)
    return  context_aware

In [9]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

vocab_size = 100000
embedding_dim = 512

embedding_matrix = np.random.randn(vocab_size, embedding_dim)


In [10]:
text = "My name is Mangesh..."

tokens = enc.encode(text)
embedding_of_words = embedding_matrix[tokens]
embedding_of_words

array([[ 1.22320461, -0.35361863,  0.7228319 , ...,  0.45088939,
        -0.90887055,  1.41232686],
       [-0.07704907,  0.24984323, -0.73480385, ..., -0.68752158,
        -0.40492411, -2.18023059],
       [ 2.02669448,  1.15234395,  0.38169707, ..., -0.0982114 ,
         0.99283947,  0.80439573],
       [ 0.39838811,  1.848772  ,  0.85568929, ...,  0.10115981,
         0.10820131,  0.12896703],
       [ 1.80292288, -0.26195043,  0.77771517, ..., -0.01877672,
        -2.78522811, -1.15481546],
       [ 3.42150254, -0.12742126, -1.58229282, ...,  0.738788  ,
         1.22648076,  0.70345109]], shape=(6, 512))

In [11]:
f = multi_head_attnetion(embedding_of_words)

In [12]:
f.shape

(6, 512)

positional encoder

In [13]:
# based on formula

def positional_encoder_f(pos, embedding_dimension):
    encoded_pos = []    
    for i in range (int(embedding_dimension / 2)) :
        a = np.sin(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        b = np.cos(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        encoded_pos += [a,b]

    return np.concatenate(np.array([encoded_pos]))

In [14]:
positional_encoder_f(1, 8)

array([0.84147098, 0.54030231, 0.09983342, 0.99500417, 0.00999983,
       0.99995   , 0.001     , 0.9999995 ])

In [15]:
def get_positional_encoding_for_fixed_number_of_words(number_of_words, embedding_dimension):
    encoded_positions = []
    for i in range (number_of_words):
        encoded_positions += [positional_encoder_f(i,embedding_dimension)]
        # print(encoded_positions, "\n")

    # print("\n\n", encoded_positions)
    return np.concatenate([encoded_positions])
    
    

In [16]:
embedding_of_words

array([[ 1.22320461, -0.35361863,  0.7228319 , ...,  0.45088939,
        -0.90887055,  1.41232686],
       [-0.07704907,  0.24984323, -0.73480385, ..., -0.68752158,
        -0.40492411, -2.18023059],
       [ 2.02669448,  1.15234395,  0.38169707, ..., -0.0982114 ,
         0.99283947,  0.80439573],
       [ 0.39838811,  1.848772  ,  0.85568929, ...,  0.10115981,
         0.10820131,  0.12896703],
       [ 1.80292288, -0.26195043,  0.77771517, ..., -0.01877672,
        -2.78522811, -1.15481546],
       [ 3.42150254, -0.12742126, -1.58229282, ...,  0.738788  ,
         1.22648076,  0.70345109]], shape=(6, 512))

In [17]:
pe = get_positional_encoding_for_fixed_number_of_words(6,512)

In [18]:
emb_with_pe = embedding_of_words + pe

In [19]:
contextually_aware  = multi_head_attnetion(emb_with_pe)

In [20]:
z =  contextually_aware + emb_with_pe

layer normalization


In [21]:
def normalize(embedding):
# 0 -> Column
# 1 -> Row
    m = np.mean(embedding, axis = 1, keepdims=True)
    s = np.std(embedding, axis = 1, keepdims=True)

    # We can improve this , have to add beta and gaamaa things
    
    return (embedding - m) / s
    

In [22]:
z_norm = normalize(z)

feed forward network


In [23]:
import torch
from torch import nn
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [24]:
class FeedForward(nn.Module):
    def __init__(self ,embedding_dimension, hidden_dimension):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(embedding_dimension, hidden_dimension)
        self.linear2 = nn.Linear(hidden_dimension, embedding_dimension)

    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return x

In [25]:
model = FeedForward(512, 2048).to(device)
print(model)

FeedForward(
  (linear1): Linear(in_features=512, out_features=2048, bias=True)
  (linear2): Linear(in_features=2048, out_features=512, bias=True)
)


In [26]:
ffr = model(torch.tensor(z_norm, dtype = torch.float32).to(device))

In [27]:
ffr.shape

torch.Size([6, 512])

In [28]:
# z_norm = torch.from_numpy(z_norm)
ffr= ffr.detach().cpu().numpy()

In [29]:
y = z_norm + ffr

In [30]:
y_norm = normalize(y)

In [31]:
y_norm.shape

(6, 512)

In [32]:
def encoder_block(input):
    contextually_aware = multi_head_attnetion(input)
    z = input + contextually_aware
    z_norm = normalize(z)
    ffr = model(torch.tensor(z_norm, dtype=torch.float32).to(device))
    ffr = ffr.detach().cpu().numpy()
    y = z_norm + ffr

    y_norm = normalize(y)
    print("Encoder block created \n")

    return y_norm

In [33]:

def encoder_architecture(input, count):
    count = count - 1
    y = encoder_block(input)
    if (count == 0):
        return y
    return encoder_architecture(y, count)

In [34]:
encoder_output = encoder_architecture(embedding_of_words, 6)

Encoder block created 

Encoder block created 

Encoder block created 

Encoder block created 

Encoder block created 

Encoder block created 



In [35]:
encoder_output.shape

(6, 512)

### Decoder Block

In [36]:
START = 100001


In [37]:
get_positional_encoding_for_fixed_number_of_words(15,512)

array([[ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00, ...,
         1.00000000e+00,  0.00000000e+00,  1.00000000e+00],
       [ 8.41470985e-01,  5.40302306e-01,  8.21856190e-01, ...,
         9.99999994e-01,  1.03663293e-04,  9.99999995e-01],
       [ 9.09297427e-01, -4.16146837e-01,  9.36414739e-01, ...,
         9.99999977e-01,  2.07326584e-04,  9.99999979e-01],
       ...,
       [-5.36572918e-01,  8.43853959e-01, -8.36262482e-01, ...,
         9.99999169e-01,  1.24395919e-03,  9.99999226e-01],
       [ 4.20167037e-01,  9.07446781e-01, -2.57667035e-02, ...,
         9.99999024e-01,  1.34762240e-03,  9.99999092e-01],
       [ 9.90607356e-01,  1.36737218e-01,  8.06904158e-01, ...,
         9.99998868e-01,  1.45128559e-03,  9.99998947e-01]],
      shape=(15, 512))

In [83]:
def masked_multi_head_attention(embedding_of_words):
    r,c = embedding_of_words.shape
    for i in range(r):
        for j in range(i+1,c):
            embedding_of_words[i][j] = 0
    
    masked_multi_head_ = multi_head_attnetion(embedding_of_words)

    return masked_multi_head_

In [84]:
masked_embedding_of_words = masked_multi_head_attention(embedding_of_words)

In [85]:
(masked_embedding_of_words)

array([[-1.54329716, -0.65615337, -3.07649561, ...,  2.08278161,
        -2.98953511, -1.60264522],
       [-1.42558226, -0.60012385, -2.99727847, ..., -0.15270302,
        -0.89434822, -0.22236162],
       [-1.54329725, -0.6561534 , -3.07649581, ...,  2.08441518,
        -2.99267627, -1.60423861],
       [-1.54329725, -0.6561534 , -3.07649581, ..., -0.28828647,
        -0.85253009, -0.17034903],
       [ 0.32256618,  0.12657699,  4.9661863 , ...,  2.08850885,
        -2.99637384, -1.60671439],
       [-1.54329725, -0.6561534 , -3.07649581, ..., -0.38228078,
        -0.7659213 , -0.11288151]], shape=(6, 512))

In [86]:
number_of_words = 6
embedding_dimension = 512

In [87]:
pos  = get_positional_encoding_for_fixed_number_of_words(number_of_words, embedding_dimension)

In [88]:
positional_and_masked = masked_embedding_of_words + pos


In [ ]:
y_normalized = normalize(positional_and_mas2
                         ked)

: 